# 04 · Everything at once: `explore()`

Train a `GeodesicSOM` and open all its views, linked: the `SOMViewer` (every layer, click-to-inspect) and the
`ComponentMatrixViewer` (one map per attribute). With live figures, drag either and both turn; select a neuron in
either and both follow. This is what `python -m mt.geodesicsom` does; here are the library calls.

Script version: [`examples/04_explore_all.py`](../04_explore_all.py) · Needs: numpy, mtgeodesicdome, matplotlib, ipywidgets
(optional: `ipympl` for live figures)

<sub>SPDX-License-Identifier: AGPL-3.0-or-later · Copyright (C) 2022-2026 Masahiro Takatsuka. See the NOTICE file for attribution terms.</sub>

## Set-up

In [ ]:
import nb_setup  # makes mt.geodesicsom importable from a checkout; helpers for the controls below

WIDGET = nb_setup.setup()  # live figures with ipympl, static images otherwise

In [ ]:
import ipywidgets as widgets
import numpy as np

from mt.geodesicsom import datasets
from mt.geodesicsom.GeodesicSOM import GeodesicSOM
from mt.geodesicsom.gui import explore, train_and_explore

## 1. Data and a spherical SOM

In [ ]:
data, labels, names = datasets.load('clusters')      # or a CSV: datasets.load('my.csv', label_column='species')
x = datasets.standardise(data)

som = GeodesicSOM(10, seed=0)                        # 1002 neurons, each with one weight per attribute
som.initialise(dataset=x)
som.train(x, epochs=30)
print(f'{som}: QE {som.quantisation_error(x):.3f}, TE {som.topographic_error(x):.3f}')

## 2. Every view, linked

> **Live or static?** With `ipympl` installed the viewers below are live: drag a sphere to rotate it, click a
> neuron to inspect it, and use the radio buttons, check boxes and keys on the figure itself. Without it they are
> images, and the controls under each viewer redraw them.

In [ ]:
views = explore(som, x, labels=labels, feature_names=names, show=False)

In [ ]:
nb_setup.controls(views.map, views.matrix, neuron=int(np.argmax(views.map.hits)))

## 3. Train and explore in one call

`train_and_explore(data, ...)` builds and trains the SOM too, and returns `(som, views)`: `lattice='sphere'` (a `GeodesicSOM` of the given
`frequency`) or `'hexagonal'` / `'rectilinear'` (a `PlaneSOM` of `rows` × `cols`, `topology='plane'` or
`'torus'`). Pick the settings and press **Run Interact**.

In [ ]:
@widgets.interact_manual(data=widgets.Dropdown(options=[*datasets.samples(), 'colours'], value='iris'),
                         lattice=widgets.ToggleButtons(options=['sphere', 'hexagonal', 'rectilinear']),
                         frequency=widgets.IntSlider(value=8, min=2, max=16, description='sphere freq'),
                         topology=widgets.ToggleButtons(options=['plane', 'torus'], description='flat'),
                         epochs=widgets.IntSlider(value=30, min=1, max=100))
def one_call(data, lattice, frequency, topology, epochs):
    x, y, names = datasets.load(data)
    with nb_setup.building():           # it standardises the data itself (standardise=True)
        som, views = train_and_explore(x, y, names, lattice=lattice, frequency=frequency, rows=16, cols=24,
                                       topology=topology, epochs=epochs, standardise=data != 'colours',
                                       verbose=False, show=False)
    if data != 'colours':
        x = datasets.standardise(x)
    print(f'{som}: QE {som.quantisation_error(x):.3f}, TE {som.topographic_error(x):.3f}')
    nb_setup.show(views.map, views.matrix)
    nb_setup.controls(views.map, views.matrix, neuron=int(np.argmax(views.map.hits)))